In [ ]:
import sqlite3, random, csv
from datetime import date, timedelta

random.seed(42)
conn = sqlite3.connect("bigbasket_capstone.db")
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS category_targets;

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    supplier TEXT NOT NULL,
    unit_price_inr INTEGER NOT NULL
);

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    signup_date TEXT NOT NULL,
    city TEXT NOT NULL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    amount_inr INTEGER NOT NULL,
    payment_mode TEXT NOT NULL,
    status TEXT NOT NULL,
    rating INTEGER,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE TABLE category_targets (
    category TEXT PRIMARY KEY,
    target_revenue_inr INTEGER NOT NULL
);
""")

cities = ["Bengaluru", "Mumbai", "Hyderabad", "Pune"]
products_raw = [
    ("Banana 1kg", "Fruits & Vegetables", "FreshFarms Co", 50),
    ("Tomato 1kg", "Fruits & Vegetables", "FreshFarms Co", 40),
    ("Onion 1kg", "Fruits & Vegetables", "GreenValley Traders", 35),
    ("Apple 1kg", "Fruits & Vegetables", "GreenValley Traders", 180),
    ("Spinach Bunch", "Fruits & Vegetables", "FreshFarms Co", 25),
    ("Toned Milk 1L", "Dairy & Eggs", "DairyBest Ltd", 60),
    ("Paneer 200g", "Dairy & Eggs", "DairyBest Ltd", 90),
    ("Eggs (12pc)", "Dairy & Eggs", "CountryEggs Farms", 84),
    ("Curd 400g", "Dairy & Eggs", "DairyBest Ltd", 45),
    ("Butter 100g", "Dairy & Eggs", "CountryEggs Farms", 55),
    ("Potato Chips 90g", "Snacks & Beverages", "SnackHub India", 30),
    ("Cola 750ml", "Snacks & Beverages", "SnackHub India", 45),
    ("Biscuit Pack", "Snacks & Beverages", "BakeHouse Supplies", 35),
    ("Fruit Juice 1L", "Snacks & Beverages", "SnackHub India", 110),
    ("Namkeen 200g", "Snacks & Beverages", "BakeHouse Supplies", 60),
    ("Shampoo 340ml", "Personal Care", "CarePlus Distributors", 220),
    ("Toothpaste 150g", "Personal Care", "CarePlus Distributors", 95),
    ("Soap Bar 125g", "Personal Care", "CarePlus Distributors", 40),
    ("Hand Wash 250ml", "Personal Care", "CarePlus Distributors", 99),
    ("Face Wash 100g", "Personal Care", "CarePlus Distributors", 150),
    ("Dish Wash Bar", "Household Essentials", "HomeEssentials Traders", 20),
    ("Detergent 1kg", "Household Essentials", "HomeEssentials Traders", 130),
    ("Floor Cleaner 1L", "Household Essentials", "HomeEssentials Traders", 145),
    ("Toilet Cleaner 500ml", "Household Essentials", "HomeEssentials Traders", 89),
    ("Garbage Bags (30pc)", "Household Essentials", "HomeEssentials Traders", 75),
    ("Bread Loaf", "Bakery", "BakeHouse Supplies", 45),
    ("Croissant (2pc)", "Bakery", "BakeHouse Supplies", 70),
    ("Muffin Pack (4pc)", "Bakery", "BakeHouse Supplies", 120),
    ("Cake Slice", "Bakery", "BakeHouse Supplies", 85),
    ("Cookies 200g", "Bakery", "BakeHouse Supplies", 65),
    ("Premium Face Cream 50g", "Personal Care", "CarePlus Distributors", 450),
]

products = [(i, *p) for i, p in enumerate(products_raw, start=1)]
cur.executemany("INSERT INTO products VALUES (?,?,?,?,?)", products)

first_names = [
    "Aarav", "Vivaan", "Aditya", "Vihaan", "Arjun", "Sai", "Reyansh", "Ayaan", "Krishna", "Ishaan",
    "Ananya", "Diya", "Saanvi", "Aadhya", "Kiara", "Myra", "Anika", "Navya", "Riya", "Siya",
    "Rohan", "Kabir", "Dev", "Yash", "Aryan", "Zara", "Meera", "Tara", "Nisha", "Priya",
    "Aman", "Rahul", "Karan", "Varun", "Nikhil", "Pooja", "Neha", "Simran", "Divya", "Isha",
    "Rohit", "Sanjay", "Vikram", "Manish", "Deepak", "Kavya", "Shreya", "Anjali", "Pallavi", "Sneha"
]

customers = []
start_signup = date(2025, 1, 1)
for i, fname in enumerate(first_names, start=1):
    city = cities[i % len(cities)]
    signup = start_signup + timedelta(days=random.randint(0, 400))
    customers.append((i, fname, signup.isoformat(), city))

cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)

popularity_weights = [10, 8, 6, 3, 5, 9, 7, 6, 8, 4, 10, 9, 5, 4, 6, 3, 5, 9, 4, 3, 8, 6, 5, 4, 7, 6, 4, 3, 5, 4]
order_date_start = date(2026, 1, 1)
order_date_end = date(2026, 6, 30)
total_days = (order_date_end - order_date_start).days
TOTAL_ORDERS = 500

product_ids_weighted = []
for pid, w in zip(range(1, 31), popularity_weights):
    product_ids_weighted.extend([pid] * w)

payment_modes = ["UPI", "Credit Card", "Debit Card", "Cash on Delivery", "Wallet"]
product_lookup = {p[0]: p for p in products}
customer_lookup = {c[0]: c for c in customers}

orders = []
order_id = 1
for _ in range(TOTAL_ORDERS):
    cust_id = random.randint(1, 50)
    prod_id = random.choice(product_ids_weighted)
    unit_price = product_lookup[prod_id][4]
    quantity = random.randint(1, 5)
    amount = quantity * unit_price
    day_offset = random.randint(0, total_days)
    o_date = order_date_start + timedelta(days=day_offset)
    payment_mode = random.choice(payment_modes)

    roll = random.random()
    if roll < 0.85:
        status = "Delivered"
        rating = random.randint(1, 5)
    elif roll < 0.95:
        status = "Cancelled"
        rating = None
    else:
        status = "Pending"
        rating = None

    orders.append([order_id, cust_id, prod_id, o_date.isoformat(), quantity, amount, payment_mode, status, rating])
    order_id += 1

cur.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?,?,?,?)", [tuple(o) for o in orders])

category_targets = [
    ("Fruits & Vegetables", 12000),
    ("Dairy & Eggs", 16500),
    ("Snacks & Beverages", 13000),
    ("Personal Care", 15500),
    ("Household Essentials", 17000),
    ("Bakery", 12000),
]
cur.executemany("INSERT INTO category_targets VALUES (?,?)", category_targets)
conn.commit()

raw_rows = []
for o in orders:
    order_id, cust_id, prod_id, o_date, qty, amt, pm, status, rating = o
    cust = customer_lookup[cust_id]
    raw_rows.append({
        "order_id": order_id,
        "order_date": o_date,
        "customer_name": cust[1],
        "city": cust[3],
        "category": product_lookup[prod_id][2],
        "product_id": prod_id,
        "quantity": qty,
        "amount_inr": amt,
        "payment_mode": pm,
        "status": status,
        "rating": rating if rating is not None else "",
    })

rng2 = random.Random(7)
casing_idx = rng2.sample(range(len(raw_rows)), 20)
for idx in casing_idx:
    r = raw_rows[idx]
    variant = rng2.choice(["upper", "lower", "space"])
    r["city"] = r["city"].upper() if variant == "upper" else (r["city"].lower() if variant == "lower" else " " + r["city"] + " ")
    variant2 = rng2.choice(["upper", "lower", "space"])
    r["category"] = r["category"].upper() if variant2 == "upper" else (r["category"].lower() if variant2 == "lower" else " " + r["category"] + " ")

null_idx = rng2.sample([i for i in range(len(raw_rows)) if i not in casing_idx], 10)
for idx in null_idx:
    raw_rows[idx]["amount_inr"] = ""

outlier_pool = [i for i in range(len(raw_rows)) if i not in casing_idx and i not in null_idx]
outlier_idx = rng2.sample(outlier_pool, 5)
for idx in outlier_idx:
    raw_rows[idx]["amount_inr"] = raw_rows[idx]["amount_inr"] * 40

dup_pool = [i for i in range(len(raw_rows)) if i not in casing_idx and i not in null_idx and i not in outlier_idx]
dup_idx = rng2.sample(dup_pool, 8)
all_rows = raw_rows + [dict(raw_rows[i]) for i in dup_idx]

fieldnames = ["order_id", "order_date", "customer_name", "city", "category", "product_id", "quantity", "amount_inr", "payment_mode", "status", "rating"]

with open("orders_raw.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(all_rows)

with open("products.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["product_id", "product_name", "category", "supplier", "unit_price_inr"])
    w.writerows(products)

conn.close()
print("bigbasket_capstone.db, orders_raw.csv, products.csv created successfully.")

bigbasket_capstone.db, orders_raw.csv, products.csv created successfully.


In [ ]:
import sqlite3

conn = sqlite3.connect("bigbasket_capstone.db")
cur = conn.cursor()

for table in ["products", "customers", "orders", "category_targets"]:
    cur.execute(f"SELECT COUNT(*) FROM {table};")
    print(f"Total rows in {table}: {cur.fetchone()[0]}")

print("\nOrders status breakdown:")
cur.execute("SELECT status, COUNT(*) FROM orders GROUP BY status;")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[1]}")

conn.close()

Total rows in products: 31
Total rows in customers: 50
Total rows in orders: 500
Total rows in category_targets: 6

Orders status breakdown:
  Cancelled: 42
  Delivered: 434
  Pending: 24


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("bigbasket_capstone.db")

# 1. SELECT / WHERE: Orders in Bengaluru
print("--- 1. Orders in Bengaluru ---")
query_1 = """
SELECT o.order_id, c.name, c.city, o.amount_inr
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE c.city = 'Bengaluru'
LIMIT 5;
"""
print(pd.read_sql_query(query_1, conn))

# 2. DISTINCT: Every distinct category
print("\n--- 2. Distinct Categories ---")
query_2 = "SELECT DISTINCT category FROM products;"
print(pd.read_sql_query(query_2, conn))

# 3. ORDER BY + LIMIT: The 5 highest-value orders
print("\n--- 3. Top 5 Highest-Value Orders ---")
query_3 = """
SELECT order_id, customer_id, product_id, amount_inr
FROM orders
ORDER BY amount_inr DESC
LIMIT 5;
"""
print(pd.read_sql_query(query_3, conn))

# 4. Alias (AS): Count total orders with alias
print("\n--- 4. Total Delivered Orders Count (Aliased) ---")
query_4 = "SELECT COUNT(*) AS total_orders FROM orders WHERE status = 'Delivered';"
print(pd.read_sql_query(query_4, conn))

# 5. IN: Orders whose payment mode is in a 2-mode list
print("\n--- 5. Orders using UPI or Credit Card ---")
query_5 = """
SELECT order_id, payment_mode, amount_inr
FROM orders
WHERE payment_mode IN ('UPI', 'Credit Card')
LIMIT 5;
"""
print(pd.read_sql_query(query_5, conn))

# 6. BETWEEN / NOT BETWEEN: Amount ranges
print("\n--- 6. Orders with Amount Between 100 and 300 INR ---")
query_6 = """
SELECT order_id, amount_inr
FROM orders
WHERE amount_inr BETWEEN 100 AND 300
LIMIT 5;
"""
print(pd.read_sql_query(query_6, conn))

# 7. IS NULL: Orders with no rating recorded
print("\n--- 7. Unrated Orders (Cancelled/Pending) ---")
query_7 = """
SELECT order_id, status, rating
FROM orders
WHERE rating IS NULL
LIMIT 5;
"""
print(pd.read_sql_query(query_7, conn))

conn.close()

--- 1. Orders in Bengaluru ---
   order_id   name       city  amount_inr
0         2  Ayaan  Bengaluru          60
1         3   Yash  Bengaluru         150
2        11   Tara  Bengaluru         300
3        16  Ayaan  Bengaluru         120
4        20   Siya  Bengaluru          45

--- 2. Distinct Categories ---
               category
0   Fruits & Vegetables
1          Dairy & Eggs
2    Snacks & Beverages
3         Personal Care
4  Household Essentials
5                Bakery

--- 3. Top 5 Highest-Value Orders ---
   order_id  customer_id  product_id  amount_inr
0       214           15          16        1100
1       396            7           4         900
2       232           39          20         750
3       288           24          23         725
4       152           22           4         720

--- 4. Total Delivered Orders Count (Aliased) ---
   total_orders
0           434

--- 5. Orders using UPI or Credit Card ---
   order_id payment_mode  amount_inr
0         1         

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("bigbasket_capstone.db")

# 1. GROUP BY & SUM: Total revenue per category
print("--- Total Revenue by Category ---")
query_rev = """
SELECT category, SUM(amount_inr) AS total_revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
WHERE o.status = 'Delivered'
GROUP BY category
ORDER BY total_revenue DESC;
"""
df_rev = pd.read_sql_query(query_rev, conn)
print(df_rev)

# 2. LEFT JOIN: Checking target coverage (including zero-order products if any)
print("\n--- Category Target vs Actual Revenue ---")
query_targets = """
SELECT
    t.category,
    t.target_revenue_inr,
    COALESCE(SUM(o.amount_inr), 0) AS actual_revenue
FROM category_targets t
LEFT JOIN products p ON t.category = p.category
LEFT JOIN orders o ON p.product_id = o.product_id AND o.status = 'Delivered'
GROUP BY t.category, t.target_revenue_inr;
"""
print(pd.read_sql_query(query_targets, conn))

# 3. Monthly Category Revenue Export (Crucial for Part 2 & 3)
query_monthly = """
SELECT
    strftime('%Y-%m', o.order_date) AS order_month,
    p.category,
    SUM(o.amount_inr) AS monthly_revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
WHERE o.status = 'Delivered'
GROUP BY order_month, p.category
ORDER BY order_month, p.category;
"""
df_monthly = pd.read_sql_query(query_monthly, conn)

# Export to CSV file
df_monthly.to_csv("monthly_category_revenue.csv", index=False)
print("\nSuccessfully exported 'monthly_category_revenue.csv'.")
print(df_monthly.head(10))

conn.close()

--- Total Revenue by Category ---
               category  total_revenue
0  Household Essentials          21715
1         Personal Care          16382
2                Bakery          15410
3          Dairy & Eggs          14090
4    Snacks & Beverages          10895
5   Fruits & Vegetables           9790

--- Category Target vs Actual Revenue ---
               category  target_revenue_inr  actual_revenue
0                Bakery               12000           15410
1          Dairy & Eggs               16500           14090
2   Fruits & Vegetables               12000            9790
3  Household Essentials               17000           21715
4         Personal Care               15500           16382
5    Snacks & Beverages               13000           10895

Successfully exported 'monthly_category_revenue.csv'.
  order_month              category  monthly_revenue
0     2026-01                Bakery              735
1     2026-01          Dairy & Eggs             2035
2     2026-01   

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the raw dataset
df = pd.read_csv("orders_raw.csv")
print(f"Initial raw row count: {len(df)}")

# 2. Clean text columns (strip trailing/leading spaces and standardize casing)
df["city"] = df["city"].astype(str).str.strip().str.title()
df["category"] = df["category"].astype(str).str.strip().str.title()

# 3. Handle missing or blank values in 'amount_inr'
# Replace empty strings or whitespace with NaN, then impute or drop appropriately
df["amount_inr"] = pd.to_numeric(df["amount_inr"], errors="coerce")

# Impute missing amounts using quantity * unit price or drop/fill strategy.
# Let's fill null amounts with the median or calculated baseline for safety:
df["amount_inr"] = df["amount_inr"].fillna(df["amount_inr"].median())

# 4. Correct numerical outliers (e.g., amounts multiplied errantly by 40)
# A reasonable threshold check based on max item price * max quantity (e.g., 450 * 5 = 2250 max normal)
q_high = df["amount_inr"].quantile(0.99)
df.loc[df["amount_inr"] > 2500, "amount_inr"] = df["amount_inr"] / 40  # correcting the factor

# 5. Drop duplicate rows introduced during raw generation
initial_count = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_count - len(df)} duplicate rows.")

# 6. Fill missing ratings with a default category or empty string if unrated
df["rating"] = df["rating"].fillna("Unrated")

# Save the cleaned dataset
df.to_csv("orders_cleaned.csv", index=False)
print("Successfully cleaned and exported 'orders_cleaned.csv'.")
print(df.info())

Initial raw row count: 508
Removed 8 duplicate rows.
Successfully cleaned and exported 'orders_cleaned.csv'.
<class 'pandas.core.frame.DataFrame'>
Index: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       500 non-null    int64  
 1   order_date     500 non-null    object 
 2   customer_name  500 non-null    object 
 3   city           500 non-null    object 
 4   category       500 non-null    object 
 5   product_id     500 non-null    int64  
 6   quantity       500 non-null    int64  
 7   amount_inr     500 non-null    float64
 8   payment_mode   500 non-null    object 
 9   status         500 non-null    object 
 10  rating         500 non-null    object 
dtypes: float64(1), int64(3), object(7)
memory usage: 46.9+ KB
None
